# Coalition Finding

Let's assume we are explanining a prediction of a model with a set of features $N$ using some value function $\nu: \mathcal{P}(N) \rightarrow \R$.
We obtain an explanation vector $e = \{e_T\}_{T\subseteq N, |T| \leq k}$ which assigns an interaction value $e_T$ to all coalitions $T$ up to some order $k \in \N$.
From this explanation vector, we can create a _simplified game_ $\hat \nu_e$ which approximates the original value function $\nu$:
$$
    \hat \nu_e(S) := \sum_{T\subseteq S, |T|\leq k} e_T
$$

$\def\min{\text{min}}
\def\max{\text{max}}$
Our goal will now be to find for some size $\ell \in \N$ the coalitions $S_\ell^\min$ and $S_\ell^\max$ that minimize and maximize $\hat \nu_e$ respectively.
An exhaustive search over all $S \subseteq N$ would have an exponential runtime complexity and is therefore not feasible for large feature sets $N$.
Instead, we turn to heuristic approaches in order to approximate $S_\ell^\min$ and $S_\ell^\max$.

## Strategy No. 1: Solos

Our first strategy is called 'Solos'.
As the name suggests, every player is considered individually: We rank all players $p \in N$ by looking only at their Shapley value $e_{\{p\}}$, ignoring all interactions of order $k \geq 2$.
We then choose the $\ell$ players with the lowest Shapley values as the minimal coalition $S_\ell^\min$ and the $\ell$ players with the highest values as $S_\ell^\max$.

This makes for a runtime of $O(n \log n)$, where $n := |N|$, since we only need to sort the array of Shapley values in order to identify the $\ell$ highest and lowest players.
(This assumes, of course, that the interaction values vector $e$ is represented in memory in such a way that all interactions of order $1$ can be retrieved without having to traverse its entirety.)

## Strategy No. 2: Equal Payoff

$\def\min{\text{min}}
\def\max{\text{max}}$
Our second strategy, called 'Equal Payoff', is slightly more sophisticated than the first one.
Instead of only considering Shapley values, we take all Shapley interactions into account.
We assign a score $s_p$ to each player $p$ by distributing the value of all interactions evenly among their respective participants.
(The baseline $e_\emptyset$ can simply be ignored since it doesn't affect $S_\ell^\min$ and $S_\ell^\max$ anyway.)
The score $s_p$ is defined as follows:
$$
    s_p := \sum_{T\subseteq S,\> p \in T} \frac{e_T}{|T|}
$$

After computing $s_p$ for all players $p \in \N$, we again sort them by their scores and choose the best and worst $\ell$ players as $S_\ell^\max$ and $S_\ell^\min$.

To compute the scores, we iterate over all subsets $T$ of order $\leq k$ and each time increase $s_p$ for all $p \in T$. Therefore the amount of operations required is
$$
    \sum_{j=1}^k \> j \binom{n}{j}
    \leq \sum_{j=1}^k \> j \cdot n^j.
$$
The complexity is decided by the largest exponent of $n$, giving us $O(n^k)$.

## Strategy No. 3: Greedy Search

This strategy is quite simple. We start by choosing the player with the lowest (respectively highest) Shapley value, then keep adding new players until we reach the desired coalition size $\ell$, each time choosing the player that increase the total value of the coalition the least (the most).

By precomputing a mapping from each player to the coalitions he's part of, we can compute the increase in payoff for a potential new player with $k \cdot n^k$ operations. This check needs to be performed for (almost) all $n$ potential new joiners in every itertaion until we've reached a size of $\ell$ players, giving us a final complexity of $O(\ell k n^{k+1}) = O(n^{k+1})$.

## Example

In this example, we'll generate a Sum of Unanimity games, which is the linear sum of a bunch of _Unanimity_ subgames. Each of the subgames is defined by a subset $U \subseteq N$ and has utility 1 iff all players of $U$ are present in the coalition.
From that, we generate an explanation using the `ExactComputer`.

In [ ]:
from shapiq import ExactComputer
from shapiq.games.benchmark import SOUM

n_players = 10
explanation_order = 3
game = SOUM(n=10, n_basis_games=50, random_state=43)
computer = ExactComputer(n_players=game.n_players, game=game)
explanation = computer(index="FSII", order=explanation_order)
explanation

Now, we can run our coalition finding algorithms on the simplified game defined by the explanation.

In [ ]:
from shapiq_student.coalition_finder import subset_finding
from shapiq_student.coalition_finder.util import get_min_max_from_interaction_values

for strategy in ["solos", "equal_payoff", "greedy"]:
    coal_size = 4
    result = subset_finding(explanation, max_size=coal_size, strategy=strategy)
    (min_coal, min_val), (max_coal, max_val) = get_min_max_from_interaction_values(result)
    # Convert from np.int to native int for better formatting
    min_coal = tuple(map(int, min_coal))
    max_coal = tuple(map(int, max_coal))
    print(f"Strategy: {strategy}, min coal. {min_coal}, max coal. {max_coal}")

It would be helpful to get an idea of how close these estimated coalitions were to the actual minimal and maximal coalitions.
For this we provide a function that will perform an exhaustive search and then rank the results of the strategy among all possible coalitions.

In [ ]:
from shapiq_student.coalition_finder.benchmark import score_single_game

for strategy in ["solos", "equal_payoff", "greedy"]:
    coal_size = 4
    n_coals, min_rank, max_rank, _ = score_single_game(
        explanation, coal_size=coal_size, strategy=strategy
    )
    print(f"Strategy: {strategy}")
    print(
        f"Total no. of coalitions {n_coals}; rank of minimal coalition {min_rank}, maximal coalition {max_rank}"
    )
    print()

These results look pretty nice, but it's actually in large part because we just got a lucky seed. In order to assess the performance of the strategies better, let's test them across a range of randomly generated games. The `shapiq_student` library provides the function `coalition_finder.benchmark` function for this, as well as a function to generate random SOUM games.

For each game, the result of the coalition will scored according to the rank of the minimal and maximal coalition among the range of all possible coalitions. Finally, these scores will be averaged over all coalitions.

In [ ]:
from shapiq_student.coalition_finder.benchmark import benchmark, random_ivs_from_soums

explanation_order = 3
for strategy in ["solos", "equal_payoff", "greedy"]:
    avg_min_score, avg_max_score, avg_t_delta = benchmark(
        strategy=strategy,
        coal_size=coal_size,
        ivs=random_ivs_from_soums(
            n_games=20,
            n_players=10,
            n_basis_games=50,
            explanation_order=explanation_order,
            random_state=52,
        ),
    )
    print(f"Strategy: {strategy}")
    print(f"Avg. score of minimal coalition: {avg_min_score:.3f}")
    print(f"Avg. score of maximal coalition: {avg_max_score:.3f}")
    print(f"Avg. time: {avg_t_delta * 1000:.3f} ms")
    print()

The scores are in the range [0, 1], with 1 being the best and 0 the worst. As we can see, the Solos strategy performs pretty poorly -- although it is quite fast. The Equal Payoff is already a bit better, but also slower. The best strategy is Greedy Search, although it is also the slowest.